# Data Description
- Background: Unit 1.3明確允許取得官方公開代碼表；本證據包保存戶政司9個官方回應的原始bytes，供人口代碼／名稱／版本查證。只新增來源，不重做人口及里界原始文件。
- date received: 20261126（本次實際取得；UTC時間逐檔記於包內manifest）
- Path to data file: https://www.ris.gov.tw/documents/html/5/1/167.html 與168.html、歷史索引558.html；每個原始網址見source_manifest.json及下方實際輸出。
- Unit of observation: ZIP成員為來源文件；VillageUTF8每列是該版本一個村里代碼／所屬區碼／名稱，並非人口或幾何。
- Sample period: 11109、11406、11409、11411、11509代碼快照；索引及RSCD0103為2026-11-26取得的現行回應，不能冒稱2024全年名冊。
- Known issues: 版本快照及穩定代碼不證明歷史幾何一致；RSCD0103含歷史／機關等條目，不以整檔作現行行政區清冊；尚無官方CPTVID轉換規範。
- Definition for each variable: `population_code`為官方村里碼（觀測11字元）；`district_code`為官方表所列所屬鄉鎮市區碼（8字元）；`official_village_name`為原表名稱。RSCD0103原行為code=name。manifest保留member、source_url、response_url、version、retrieved_at_utc、bytes、sha256、content_type、http_last_modified及不作生效日解釋註記。
- Data path: `data/raw/unit13_official_code_evidence.zip`；[來源包](unit13_official_code_evidence.zip)
- Role: `raw`（官方代碼查證來源包，非人口分析資料）
- Verification status: 以本Notebook實際執行輸出為準；逐成員hash、語系、欄位及代碼唯一性檢查。
- SHA-256: `9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40`
- Next processing notebook: [人口驗證及crosswalk](../code/taipei_population_validation.ipynb)

Author：詹明翰、謝子詮、曾翊涵（沿用已確認共同作者，不推定個別分工）。Created／last modified：2026-11-26。
ZIP是本地保存容器，並非政府直接發布的ZIP檔；其中9份原始回應bytes未改寫。


In [1]:
from pathlib import Path
import hashlib,json,sys,platform,os,io,zipfile,re
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def project_root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the student project containing AGENTS.md and templates/.')
ROOT=project_root()
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
assert sys.prefix!=sys.base_prefix
os.environ['PROJ_NETWORK']='OFF'
def file(relative,area='data',must_exist=True):
    rel=Path(relative)
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Explicit project-relative path required')
    path=(ROOT/rel).resolve()
    if not path.is_relative_to((ROOT/area).resolve()):raise ValueError('Path outside intended project area')
    if must_exist and not path.is_file():raise FileNotFoundError(relative)
    return path
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for b in iter(lambda:stream.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def verify(expected):
    rows=[]
    for relative,digest in expected.items():
        path=file(relative)
        assert sha(path)==digest,f'Input changed; stop and inspect version: {relative}'
        rows.append({'path':relative,'bytes':path.stat().st_size,'sha256':digest})
    return pd.DataFrame(rows)
def show(title,table,block=20):
    display(Markdown('**'+title+'**'))
    if not len(table):display(Markdown('無符合紀錄。'))
    for start in range(0,len(table),block):display(table.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
print({'python':platform.python_version(),'interpreter_is_project_venv':True,'network_in_notebook':'none'})

EXPECTED_SHA256={'data/raw/unit13_official_code_evidence.zip':'9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40'}
show('來源包hash',verify(EXPECTED_SHA256))


{'python': '3.12.2', 'interpreter_is_project_venv': True, 'network_in_notebook': 'none'}


**來源包hash**

,path,bytes,sha256
0,data/raw/unit13_official_code_evidence.zip,300990,9f27bd2f9211f4c21253876529efec1414b82f0b7014f6d4028b70f66b58cc40


## Change for Your Own Data
原始ZIP只讀，不解壓寫回raw；不連網、不替換來源。ZIP member依精確名稱讀入記憶體。


In [2]:
archive=zipfile.ZipFile(file('data/raw/unit13_official_code_evidence.zip'))
manifest=json.loads(archive.read('source_manifest.json').decode('utf-8'))
assert set(archive.namelist())=={r['member'] for r in manifest}|{'source_manifest.json'}
for row in manifest:
    payload=archive.read(row['member'])
    assert len(payload)==row['bytes'] and hashlib.sha256(payload).hexdigest()==row['sha256']
    assert row['source_url'].startswith('https://www.ris.gov.tw/')
show('原始來源、網址、版本與逐檔hash',pd.DataFrame(manifest),block=5)
TABLE_NAMES=['11109VillageUTF8.txt','11406VillageUTF8.txt','11409VillageUTF8.txt','11411VillageUTF8.txt','11509VillageUTF8.txt']
tables={name:pd.read_csv(io.BytesIO(archive.read(name)),header=None,
            names=['population_code','district_code','official_village_name'],dtype='string',encoding='utf-8-sig',keep_default_na=False)
        for name in TABLE_NAMES}
district_text=archive.read('RSCD0103.txt').decode('cp950',errors='strict')
print({'VillageUTF8_encoding':'UTF-8-SIG strict','RSCD0103_encoding':'CP950 strict',
       'official_catalog_RSCD0103':'省市縣市鄉鎮市區代碼 X(08)',
       '8_plus_3_relationship':'actual table prefix relationship checked below; not a published CPTVID conversion'})


**原始來源、網址、版本與逐檔hash**

,member,source_url,response_url,version,retrieved_at_utc,bytes,sha256,content_type,http_last_modified,http_last_modified_is_geometry_effective_date
0,ris_168.html,https://www.ris.gov.tw/documents/html/5/1/168.html,https://www.ris.gov.tw/documents/html/5/1/168.html,retrieved 2026-11-26; code catalog,2026-11-26T06:24:40.464960+00:00,39305,ad1152ddbb6d5c01c41fffb28f079dae3566bc72044ee529611d6e054fd59938,text/html; charset=UTF-8,"Wed, 28 Jan 2026 09:26:45 GMT",False
1,ris_167.html,https://www.ris.gov.tw/documents/html/5/1/167.html,https://www.ris.gov.tw/documents/html/5/1/167.html,retrieved 2026-11-26; current code index,2026-11-26T06:24:40.571952+00:00,24526,bf879d3314c871ec4260677802f9d3bd6a5c827f7b78a0a283bdd58e3e244c63,text/html; charset=UTF-8,"Fri, 04 Sep 2026 13:00:41 GMT",False
2,ris_558.html,https://www.ris.gov.tw/documents/html/5/1/558.html,https://www.ris.gov.tw/documents/html/5/1/558.html,retrieved 2026-11-26; historical code index,2026-11-26T06:24:40.659624+00:00,63098,1fb066026a51789a3e1c36738565d67718060f005bf333a3ab1d6f2f0cd7d7e6,text/html; charset=UTF-8,"Fri, 04 Sep 2026 11:27:12 GMT",False
3,RSCD0103.txt,https://www.ris.gov.tw/documents/data/5/1/RSCD0103.txt,https://www.ris.gov.tw/documents/data/5/1/RSCD0103.txt,"current; official town code table, no effective-date assertion",2026-11-26T06:24:40.716300+00:00,16076,c89a4bfc564aa3fa5c9c59fd0656a52c8b092c62b812f1348ac8342f7b4945e6,text/plain; charset=UTF-8,"Wed, 12 Jun 2019 11:12:15 GMT",False
4,11109VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11109VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11109VillageUTF8.txt,11109 / 2022-09,2026-11-26T06:24:40.906828+00:00,240272,78a53459ac359e2e1c79e29ea32751b5fd46e8241fa068dcd7f6fd41108e9242,text/plain; charset=UTF-8,"Fri, 12 Aug 2022 04:20:00 GMT",False


,member,source_url,response_url,version,retrieved_at_utc,bytes,sha256,content_type,http_last_modified,http_last_modified_is_geometry_effective_date
5,11406VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11406VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11406VillageUTF8.txt,11406 / 2025-06,2026-11-26T06:24:41.100767+00:00,240303,99405e9a3e62004d341cdd17d4a640eed49be1deddf48d5b92c060469b4ae0e4,text/plain; charset=UTF-8,"Fri, 13 Jun 2025 10:54:46 GMT",False
6,11409VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11409VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11409VillageUTF8.txt,11409 / 2025-09,2026-11-26T06:24:41.197444+00:00,240427,fc898ed6a6570adc4110cd54a6e75233a5996332dda9735a2b8889982d3d9b12,text/plain; charset=UTF-8,"Fri, 05 Sep 2025 08:46:02 GMT",False
7,11411VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11411VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11411VillageUTF8.txt,11411 / 2025-11,2026-11-26T06:24:41.437924+00:00,240954,740c39e4c21e7fcc8fddf9741eee26c85724558f8cee0c1f357e67ee252ed484,text/plain; charset=UTF-8,"Fri, 28 Nov 2025 12:50:05 GMT",False
8,11509VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11509VillageUTF8.txt,https://www.ris.gov.tw/documents/data/5/1/11509VillageUTF8.txt,11509 / 2026-09,2026-11-26T06:24:41.704743+00:00,241310,c7241fd8f93bc7d330e4fc52b89365a8fa7a8f695bb09012dce01c5dfbfa24b6,text/plain; charset=UTF-8,"Fri, 04 Sep 2026 13:00:41 GMT",False


{'VillageUTF8_encoding': 'UTF-8-SIG strict', 'RSCD0103_encoding': 'CP950 strict', 'official_catalog_RSCD0103': '省市縣市鄉鎮市區代碼 X(08)', '8_plus_3_relationship': 'actual table prefix relationship checked below; not a published CPTVID conversion'}


## Summary
### Sample
各來源表實際前十列；不裁切或更正原始文件。


In [3]:
for name,table in tables.items():show(name+'：原始前十列',table.head(10))
show('RSCD0103原文前十行',pd.DataFrame({'original_line':district_text.splitlines()[:10]}))


**11109VillageUTF8.txt：原始前十列**

,population_code,district_code,official_village_name
0,09007010001,09007010,介壽村
1,09007010002,09007010,復興村
2,09007010003,09007010,福沃村
3,09007010004,09007010,清水村
4,09007010005,09007010,仁愛村
5,09007010006,09007010,津沙村
6,09007010007,09007010,馬祖村
7,09007010008,09007010,珠螺村
8,09007010009,09007010,四維村
9,09007020001,09007020,塘岐村


**11406VillageUTF8.txt：原始前十列**

,population_code,district_code,official_village_name
0,09007010001,09007010,介壽村
1,09007010002,09007010,復興村
2,09007010003,09007010,福沃村
3,09007010004,09007010,清水村
4,09007010005,09007010,仁愛村
5,09007010006,09007010,津沙村
6,09007010007,09007010,馬祖村
7,09007010008,09007010,珠螺村
8,09007010009,09007010,四維村
9,09007020001,09007020,塘岐村


**11409VillageUTF8.txt：原始前十列**

,population_code,district_code,official_village_name
0,09007010001,09007010,介壽村
1,09007010002,09007010,復興村
2,09007010003,09007010,福沃村
3,09007010004,09007010,清水村
4,09007010005,09007010,仁愛村
5,09007010006,09007010,津沙村
6,09007010007,09007010,馬祖村
7,09007010008,09007010,珠螺村
8,09007010009,09007010,四維村
9,09007020001,09007020,塘岐村


... (2 outputs left out)

**11509VillageUTF8.txt：原始前十列**

,population_code,district_code,official_village_name
0,09007010001,09007010,介壽村
1,09007010002,09007010,復興村
2,09007010003,09007010,福沃村
3,09007010004,09007010,清水村
4,09007010005,09007010,仁愛村
5,09007010006,09007010,津沙村
6,09007010007,09007010,馬祖村
7,09007010008,09007010,珠螺村
8,09007010009,09007010,四維村
9,09007020001,09007020,塘岐村


**RSCD0103原文前十行**

,original_line
0,00000000=中華民國內政部
1,10000000=臺灣省
2,10000006=臺灣省兵役處
3,10001000=臺灣省臺北縣
4,10001006=臺灣省臺北縣民政局
5,10001010=臺灣省臺北縣板橋市
6,10001011=臺灣省臺北縣板橋市
7,10001012=臺灣省臺北縣板橋市
8,10001020=臺灣省臺北縣三重市
9,10001021=臺灣省臺北縣三重市


### Summary Stats
只描述代碼來源，不把全國歷史條目數當作現行里數。


In [4]:
rows=[]
for name,table in tables.items():
    taipei=table.loc[table.population_code.str.startswith('63000')]
    rows.append({'member':name,'national_rows':len(table),'columns':len(table.columns),
                 'duplicate_village_codes':int(table.population_code.duplicated().sum()),
                 'missing_or_blank_cells':int(table.eq('').to_numpy().sum()),'taipei_rows':len(taipei),
                 'village_code_types':str(table.population_code.dtype),'district_code_types':str(table.district_code.dtype)})
    assert table.population_code.is_unique
    assert table.population_code.str[:8].eq(table.district_code).all()
    assert table.population_code.str.len().eq(11).all() and table.district_code.str.len().eq(8).all()
show('來源表結構及唯一性',pd.DataFrame(rows))
verify(EXPECTED_SHA256)
print({'archive_and_all_9_source_members_unchanged':True,'existing_raw_files_modified':False})


**來源表結構及唯一性**

,member,national_rows,columns,duplicate_village_codes,missing_or_blank_cells,taipei_rows,village_code_types,district_code_types
0,11109VillageUTF8.txt,7748,3,0,0,456,string,string
1,11406VillageUTF8.txt,7749,3,0,0,456,string,string
2,11409VillageUTF8.txt,7753,3,0,0,456,string,string
3,11411VillageUTF8.txt,7770,3,0,0,456,string,string
4,11509VillageUTF8.txt,7781,3,0,0,456,string,string


{'archive_and_all_9_source_members_unchanged': True, 'existing_raw_files_modified': False}
